# Оконные функции — задание 22

В ноутбуке решены все 3 пункта задания.

Во всех графиках спектр показан в децибелах относительно максимума на графике:
`L(f) = 20 * log10(A(f) / A_max)`.

То есть значение `0 dB` соответствует самому большому пику, а все остальные компоненты показаны ниже него.


## Замечание по пунктам 2 и 3

Фраза «один период сигнала, состоящего из двух синусоид» неоднозначна, потому что сумма синусоид с частотами 10.1 Гц и 12.3 Гц (или 15.2 Гц) не имеет короткого общего периода.

Поэтому в решении длительность записи для пунктов 2 и 3 выбрана равной одному периоду основной гармоники 10.1 Гц:
`T = 1 / 10.1`.

Это стандартное допущение для задач на оконные функции и спектральную утечку.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 4.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 12


In [ ]:
def make_window(name: str, n: int) -> np.ndarray:
    name = name.lower()
    if n <= 1:
        return np.ones(n, dtype=float)

    if name in {'rect', 'rectangular', 'boxcar', 'прямоугольное'}:
        return np.ones(n, dtype=float)
    if name in {'hamming', 'хэмминга'}:
        return np.hamming(n)
    if name in {'blackman', 'блэкмана'}:
        return np.blackman(n)
    if name in {'blackman-harris', 'blackmanharris', 'блэкмана-харриса'}:
        k = np.arange(n)
        a0, a1, a2, a3 = 0.35875, 0.48829, 0.14128, 0.01168
        phase = 2 * np.pi * k / (n - 1)
        return a0 - a1 * np.cos(phase) + a2 * np.cos(2 * phase) - a3 * np.cos(3 * phase)
    raise ValueError(f'Неизвестное окно: {name}')


def generate_signal(fs: float, duration: float, frequencies, amplitudes=None, phases=None):
    frequencies = np.asarray(frequencies, dtype=float)
    amplitudes = np.ones_like(frequencies) if amplitudes is None else np.asarray(amplitudes, dtype=float)
    phases = np.zeros_like(frequencies) if phases is None else np.asarray(phases, dtype=float)

    n = max(2, int(round(fs * duration)))
    t = np.arange(n) / fs
    signal = np.zeros(n, dtype=float)
    for amp, freq, phase in zip(amplitudes, frequencies, phases):
        signal += amp * np.sin(2 * np.pi * freq * t + phase)
    return t, signal


def magnitude_spectrum_db(signal, fs: float, window_name: str, n_fft: int = 16384, floor_db: float = -160.0):
    window = make_window(window_name, len(signal))
    spectrum = np.fft.rfft(signal * window, n=n_fft)
    freqs = np.fft.rfftfreq(n_fft, d=1 / fs)

    amplitude = np.abs(spectrum) / np.sum(window)
    if len(amplitude) > 2:
        amplitude[1:-1] *= 2

    peak = np.max(amplitude)
    if peak <= 0:
        peak = 1.0

    db = 20 * np.log10(np.maximum(amplitude / peak, 10 ** (floor_db / 20)))
    return freqs, db


def plot_windows(signal, fs, window_names, title, frequencies_to_mark, xlim, ylim=(-160, 5), n_fft=16384):
    fig, axes = plt.subplots(1, len(window_names), figsize=(6 * len(window_names), 4.5), sharey=True)
    if len(window_names) == 1:
        axes = [axes]

    for ax, window_name in zip(axes, window_names):
        freqs, db = magnitude_spectrum_db(signal, fs, window_name, n_fft=n_fft, floor_db=ylim[0])
        ax.plot(freqs, db, lw=2)
        for freq in frequencies_to_mark:
            ax.axvline(freq, color='crimson', linestyle='--', linewidth=1, alpha=0.8)
        ax.set_title(window_name)
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        ax.set_xlabel('Частота, Гц')
        ax.set_ylabel('Уровень, dB')

    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


## 1. Синусоида 10.1 Гц, две длины периода, fs = 500 Гц

Берём длительность записи `T = 2 / 10.1` и строим спектры для окон:
- прямоугольное
- Хэмминга
- Блэкмана


In [ ]:
fs_1 = 500
f0_1 = 10.1
duration_1 = 2 / f0_1

t_1, x_1 = generate_signal(fs_1, duration_1, frequencies=[f0_1])
print(f'Пункт 1: N = {len(x_1)} отсчётов, фактическая длительность = {len(x_1) / fs_1:.6f} с')

plot_windows(
    signal=x_1,
    fs=fs_1,
    window_names=['Rectangular', 'Hamming', 'Blackman'],
    title='Пункт 1: спектр синусоиды 10.1 Гц для разных окон',
    frequencies_to_mark=[f0_1],
    xlim=(0, 60),
    ylim=(-140, 5),
)


Вывод: прямоугольное окно даёт самые высокие боковые лепестки. Окна Хэмминга и Блэкмана уменьшают спектральную утечку, а окно Блэкмана делает фон наиболее чистым, но расширяет главный лепесток.


## 2. Сигнал из двух синусоид 10.1 Гц и 12.3 Гц, fs = 1000 Гц

Берём длительность записи `T = 1 / 10.1` и сравниваем окна:
- прямоугольное
- Блэкмана-Харриса


In [ ]:
fs_2 = 1000
frequencies_2 = [10.1, 12.3]
duration_2 = 1 / 10.1

t_2, x_2 = generate_signal(fs_2, duration_2, frequencies=frequencies_2)
print(f'Пункт 2: N = {len(x_2)} отсчётов, фактическая длительность = {len(x_2) / fs_2:.6f} с')

plot_windows(
    signal=x_2,
    fs=fs_2,
    window_names=['Rectangular', 'Blackman-Harris'],
    title='Пункт 2: две близкие синусоиды 10.1 Гц и 12.3 Гц',
    frequencies_to_mark=frequencies_2,
    xlim=(0, 40),
    ylim=(-160, 5),
)


Вывод: окно Блэкмана-Харриса лучше подавляет боковые лепестки, поэтому утечка меньше. Но из-за более широкого главного лепестка две близкие частоты при короткой записи разделяются не идеально.


## 3. Сигнал из двух синусоид 10.1 Гц и 15.2 Гц с амплитудами 1 и 0.025, fs = 1000 Гц

Отношение амплитуд слабой и сильной гармоники равно `0.025`, что соответствует примерно `-32.04 dB`.

Сравниваем окна:
- прямоугольное
- Блэкмана


In [ ]:
fs_3 = 1000
frequencies_3 = [10.1, 15.2]
amplitudes_3 = [1.0, 0.025]
duration_3 = 1 / 10.1

t_3, x_3 = generate_signal(fs_3, duration_3, frequencies=frequencies_3, amplitudes=amplitudes_3)
print(f'Пункт 3: N = {len(x_3)} отсчётов, фактическая длительность = {len(x_3) / fs_3:.6f} с')

plot_windows(
    signal=x_3,
    fs=fs_3,
    window_names=['Rectangular', 'Blackman'],
    title='Пункт 3: сильная и слабая гармоники 10.1 Гц и 15.2 Гц',
    frequencies_to_mark=frequencies_3,
    xlim=(0, 45),
    ylim=(-160, 5),
)


Вывод: слабую гармонику на уровне около `-32 dB` удобнее наблюдать с окном Блэкмана, потому что оно сильнее подавляет боковые лепестки основной компоненты.


## Общий итог

- Все спектры построены в dB.
- Более гладкие окна уменьшают спектральную утечку.
- За уменьшение боковых лепестков приходится платить расширением главного лепестка и потерей частотного разрешения.
